# 01 - Trích xuất Đặc trưng Thủ công và Logistic Regression

Trong bài học này, chúng ta xây dựng mô hình baseline phân loại cặp ảnh:
- Trực quan hóa các phép biến đổi ảnh và tính toán vector đặc trưng trên một cặp ảnh thật từ tập TRAIN Fold 0.
- Tìm hiểu ý nghĩa của bảng 32 đặc trưng thống kê số học.
- Quy ước sai khác Right-Minus-Left và nguyên lý bảo toàn tính đối xứng $p(-\mathbf{x}) = 1 - p(\mathbf{x})$.
- Khám phá dữ liệu (EDA) và đặt giả thuyết heuristic chỉ trên tập TRAIN của Fold 0.
- Huấn luyện và so sánh trên cùng tập Validation Fold 0: Heuristic Rule, Filesize-only LR, LR32 đầy đủ, và LR32 không dùng dung lượng.

In [ ]:
from pathlib import Path
import os
import sys

def find_task_root():
    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent, cwd / 'KeMaoDanh', cwd.parent / 'KeMaoDanh']:
        if (cand / 'src/kmd').is_dir() and (cand / 'configs').is_dir():
            return cand.resolve()
    raise FileNotFoundError("Mở notebook từ repo root, KeMaoDanh hoặc KeMaoDanh/notebooks.")

TASK_ROOT = find_task_root()
if str(TASK_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(TASK_ROOT / 'src'))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from kmd.core import PACKAGE, read_csv, split_fold, metric
from kmd.extractor import NAMES, features
from kmd.pipeline import prepare_development, start_session, fit_lr_fold, extract_pair_features

# Tên phiên làm việc dùng chung cho chuỗi bài học
RUN_ID = 'lesson_session'
print(f"Phiên làm việc: {RUN_ID}")

## 1. Trực quan hóa đặc trưng và bảng vector trên một cặp ảnh thật

Chúng ta lấy một cặp ảnh từ tập TRAIN của Fold 0 để quan sát các phép biến đổi:
1. Dung lượng file: $\text{log\_bytes} = \ln(1 + \text{bytes})$.
2. Ảnh xám (grayscale).
3. Độ lớn Gradient Sobel ($G = \sqrt{G_x^2 + G_y^2}$).
4. Toán tử Laplacian đo độ biến thiên bậc 2.
5. Nhiễu dư (Residual): lấy ảnh xám trừ đi ảnh đã làm mờ Gaussian ($\sigma = 1.2$).
6. Vùng trung tâm 60% vs Vùng viền.

Gradient lớn tại nơi độ sáng đổi nhanh, chẳng hạn biên tóc. Laplacian đo biến thiên của sự thay đổi đó. Residual giữ phần ảnh bị phép làm mờ loại đi; nó có thể chứa chi tiết, nhiễu hoặc dấu vết xử lý, không tự động là bằng chứng giả mạo. Entropy tóm tắt độ phân tán histogram mức xám. Các đặc trưng trung tâm/viền kiểm tra khác biệt theo vùng; bước nhảy ở lưới 8 pixel chỉ là thống kê, không chứng minh ảnh đã bị sửa.

In [ ]:
data_root_env = os.environ.get('DATA_ROOT')
data_root = Path(data_root_env or TASK_ROOT / 'data/train').expanduser().resolve()
if not (data_root / 'pairs.csv').is_file():
    raise FileNotFoundError(f'Thiếu dữ liệu train: {data_root / "pairs.csv"}. Xem README để đặt DATA_ROOT.')

if (data_root / 'pairs.csv').is_file():
    dev_frame = prepare_development(data_root)
    tr_fold0, va_fold0 = split_fold(dev_frame, fold=0)
    sample_row = tr_fold0.iloc[0]
    
    p0 = data_root / sample_row.image_0
    p1 = data_root / sample_row.image_1
    
    # Đọc và trực quan hóa ảnh 0
    bgr0 = cv2.imread(str(p0))
    bgr1 = cv2.imread(str(p1))
    
    if bgr0 is not None and bgr1 is not None:
        rgb0 = cv2.cvtColor(bgr0, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        rgb1 = cv2.cvtColor(bgr1, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        gray0 = cv2.cvtColor(bgr0, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
        
        gx = cv2.Sobel(gray0, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray0, cv2.CV_32F, 0, 1, ksize=3)
        grad_mag = np.sqrt(gx**2 + gy**2)
        lap = np.abs(cv2.Laplacian(gray0, cv2.CV_32F))
        blurred = cv2.GaussianBlur(gray0, (0, 0), sigmaX=1.2, sigmaY=1.2)
        residual = np.abs(gray0 - blurred)
        
        h, w = gray0.shape
        mask_center = np.zeros_like(gray0)
        mask_center[int(0.2*h):int(0.8*h), int(0.2*w):int(0.8*w)] = 1.0
        
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        axes[0, 0].imshow(rgb0); axes[0, 0].set_title("image_0 (RGB)"); axes[0, 0].axis('off')
        axes[0, 1].imshow(rgb1); axes[0, 1].set_title("image_1 (RGB)"); axes[0, 1].axis('off')
        axes[0, 2].imshow(grad_mag, cmap='magma'); axes[0, 2].set_title("Sobel Gradient (image_0)"); axes[0, 2].axis('off')
        axes[1, 0].imshow(lap, cmap='inferno'); axes[1, 0].set_title("Laplacian (image_0)"); axes[1, 0].axis('off')
        axes[1, 1].imshow(residual, cmap='viridis'); axes[1, 1].set_title("Residual (image_0)"); axes[1, 1].axis('off')
        axes[1, 2].imshow(mask_center, cmap='bone'); axes[1, 2].set_title("Center 60% Mask"); axes[1, 2].axis('off')
        axes[0, 3].imshow(gray0, cmap='gray'); axes[0, 3].set_title('Grayscale'); axes[0, 3].axis('off')
        axes[1, 3].imshow(blurred, cmap='gray'); axes[1, 3].set_title('Gaussian blur'); axes[1, 3].axis('off')
        fig.suptitle(f"Cặp mẫu {sample_row['pair_id']} (Nhãn: ảnh {sample_row['fake_position']} là giả)", fontsize=13)
        plt.tight_layout()
        plt.show()
        
        # Trích xuất 32 đặc trưng của cả 2 ảnh và tính hiệu
        f0 = features(p0)
        f1 = features(p1)
        diff_feat = f1 - f0
        
        df_feats = pd.DataFrame({
            'Feature': NAMES,
            'f0 (image_0)': f0,
            'f1 (image_1)': f1,
            'x = f1 - f0': diff_feat
        })
        print(f"Kích thước vector đặc trưng mỗi ảnh: {f0.shape}, vector hiệu: {diff_feat.shape}")
        print("\nBảng một số đặc trưng tiêu biểu của cặp ảnh:")
        print(df_feats.iloc[[0, 1, 4, 13, 17, 19, 20, 24, 26, 28]].to_string(index=False))

## 2. Bảng 32 đặc trưng thống kê số học

32 đặc trưng được tính toán tuần tự cho mỗi file ảnh:
- **0:** `log_bytes` = $\ln(1 + \text{bytes})$
- **1–12:** Thống kê RGB: `mean`, `std`, `q10`, `q90` cho 3 kênh ($4 \times 3 = 12$)
- **13–16:** Thống kê HSV: `sat_mean`, `sat_std`, `val_mean`, `val_std`
- **17–19:** Ảnh xám & Entropy: `gray_mean`, `gray_std`, `entropy` (histogram 64 bins)
- **20–23:** Gradient & Laplacian: `grad_mean`, `grad_std`, `lap_std`, `lap_abs`
- **24–25:** Nhiễu dư (Residual): `residual_std`, `residual_abs`
- **26–27:** Bước nhảy lưới JPEG $8 \times 8$: `block_x`, `block_y`
- **28–31:** Vùng trung tâm vs Viền: `center_mean/std`, `border_mean/std`

## 3. Quy ước sai khác và Tính đối xứng (Symmetry)

Với vector đặc trưng $\mathbf{f}_0, \mathbf{f}_1 \in \mathbb{R}^{32}$, vector đầu vào của mô hình phân loại là hiệu số:
$$\mathbf{x} = \mathbf{f}_1 - \mathbf{f}_0$$

Nếu hoán đổi vị trí hai ảnh, vector hiệu đổi dấu: $\mathbf{x}_{\text{swap}} = \mathbf{f}_0 - \mathbf{f}_1 = -\mathbf{x}$.

Trong Logistic Regression với xác suất $p(\mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x})$ (không có hệ số chặn $b = 0$ và `StandardScaler(with_mean=False)` không dịch chuyển tâm):
$$p(-\mathbf{x}) = \sigma(\mathbf{w}^T(-\mathbf{x})) = \frac{1}{1 + e^{\mathbf{w}^T \mathbf{x}}} = 1 - \frac{1}{1 + e^{-\mathbf{w}^T \mathbf{x}}} = 1 - p(\mathbf{x})$$

Tính chất này buộc xác suất đổi nhất quán khi đổi chỗ hai ảnh. Nó không bảo đảm mô hình chính xác hay loại bỏ thiên lệch của dữ liệu. Ta sẽ kiểm tra trên LR đã fit ở cuối bài.

In [ ]:
import inspect
print(inspect.getsource(fit_lr_fold))
# Scaler và LR chỉ fit các hàng inner_fold != 0; validation không tham gia fit.

## 4. Khám phá Dữ liệu (EDA) trên tập TRAIN Fold 0

Ta kiểm tra giả thuyết: Liệu ảnh giả mạo có xu hướng có dung lượng file nặng hơn hay nhẹ hơn do thuật toán chỉnh sửa/nén?

In [ ]:
if (data_root / 'pairs.csv').is_file():
    train_bytes_diff = []
    train_labels = []
    for r in tr_fold0.itertuples():
        s0 = (data_root / r.image_0).stat().st_size
        s1 = (data_root / r.image_1).stat().st_size
        train_bytes_diff.append(np.log1p(s1) - np.log1p(s0))
        train_labels.append(r.fake_position)
        
    train_bytes_diff = np.array(train_bytes_diff)
    train_labels = np.array(train_labels)
    
    # Chia bin chung cho cả 2 lớp
    shared_bins = np.linspace(train_bytes_diff.min(), train_bytes_diff.max(), 35)
    
    plt.figure(figsize=(9, 4))
    plt.hist(train_bytes_diff[train_labels == 0], bins=shared_bins, alpha=0.6, label='Nhãn 0 (Ảnh trái giả)', color='blue')
    plt.hist(train_bytes_diff[train_labels == 1], bins=shared_bins, alpha=0.6, label='Nhãn 1 (Ảnh phải giả)', color='red')
    plt.axvline(0, color='black', linestyle='--', label='Ngưỡng x = 0')
    plt.title("Phân bố hiệu dung lượng log_bytes trên tập TRAIN Fold 0 (Shared Bins)")
    plt.xlabel("log_bytes(image_1) - log_bytes(image_0)")
    plt.ylabel("Số lượng mẫu")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 5. Đánh giá trên cùng tập Validation của Fold 0

Chúng ta đánh giá 4 phương pháp trên cùng các mẫu **Validation Fold 0**:
1. **Heuristic Rule:** Nếu $\Delta \text{log\_bytes} < 0$ thì dự đoán $1$, ngược lại dự đoán $0$ (hòa thì đoán 0).
2. **Filesize-only LR:** Mô hình Logistic Regression chỉ dùng 1 đặc trưng `log_bytes`.
3. **LR32 đầy đủ:** Logistic Regression dùng toàn bộ 32 đặc trưng.
4. **LR32 không dùng Filesize:** Logistic Regression dùng 31 đặc trưng thị giác (bỏ `log_bytes`).

Quy tắc cố định ở đây là “ảnh nhỏ hơn về số byte là ảnh giả”, hòa chọn ảnh 0. Đây là giả thuyết đơn giản để kiểm tra, không phải thuộc tính tất yếu của ảnh giả. Không đổi hướng quy tắc sau khi xem validation. Quy tắc chỉ cho nhãn; vì vậy không gán xác suất 0.9/0.1 và không báo log-loss của nó.

In [ ]:
if (data_root / 'pairs.csv').is_file():
    session_dir = start_session(dev_frame, data_root, run_id=RUN_ID)
    print(f"Lưu session tại: {session_dir.name}")
    
    # Trích xuất ma trận đặc trưng cho 800 cặp development
    x_all, feature_table = extract_pair_features(dev_frame, data_root)
    feature_table.to_csv(session_dir / 'features.csv')
    
    y_val = va_fold0.fake_position.to_numpy()
    
    # 1. Heuristic Rule trên Validation Fold 0
    va_bytes_diff = []
    for r in va_fold0.itertuples():
        s0 = (data_root / r.image_0).stat().st_size
        s1 = (data_root / r.image_1).stat().st_size
        va_bytes_diff.append(np.log1p(s1) - np.log1p(s0))
    va_bytes_diff = np.array(va_bytes_diff)
    from sklearn.metrics import f1_score
    rule_labels = (va_bytes_diff < 0).astype(int)  # bằng nhau -> 0
    score_rule = dict(macro_f1=f1_score(y_val, rule_labels, average='macro', labels=[0, 1], zero_division=0),
                      accuracy=float(np.mean(rule_labels == y_val)),
                      errors=int(np.sum(rule_labels != y_val)), log_loss=np.nan)
    
    # 2. Filesize-only LR (Cột 0)
    _, p_size = fit_lr_fold(x_all[:, [0]], dev_frame, fold=0)
    score_size = metric(y_val, p_size)
    
    # 3. LR32 đầy đủ (32 cột)
    m32, p_32 = fit_lr_fold(x_all, dev_frame, fold=0)
    score_32 = metric(y_val, p_32)
    
    # 4. LR32 without filesize (31 cột)
    _, p_no_size = fit_lr_fold(x_all[:, 1:], dev_frame, fold=0)
    score_no_size = metric(y_val, p_no_size)
    
    # Bảng tổng hợp trên cùng tập Validation Fold 0
    res_table = pd.DataFrame([
        {'Phương pháp': '1. Heuristic File-size Rule', **score_rule},
        {'Phương pháp': '2. Filesize-only LR (1 feat)', **score_size},
        {'Phương pháp': '3. LR32 Full (32 feats)', **score_32},
        {'Phương pháp': '4. LR32 without Filesize (31 feats)', **score_no_size},
    ])
    print("=== SO SÁNH CÁC BIẾN THỂ TRÊN CÙNG TẬP VALIDATION FOLD 0 ===")
    print(res_table[['Phương pháp', 'macro_f1', 'accuracy', 'log_loss', 'errors']].to_string(index=False))

In [ ]:
# Kiểm tra trên LR32 vừa fit, không gán trọng số giả.
x_sample = diff_feat.reshape(1, -1)
p_orig = m32.predict_proba(x_sample)[0, 1]
p_swap = m32.predict_proba(-x_sample)[0, 1]
print({'p_original': p_orig, 'p_swapped': p_swap, 'sum': p_orig + p_swap})
assert np.isclose(p_orig + p_swap, 1.0, atol=1e-10)

import matplotlib.pyplot as plt
from PIL import Image

def show_case(row, caption):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for side, ax in enumerate(axes):
        with Image.open(data_root / row[f'image_{side}']) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f'image_{side}')
        ax.axis('off')
    fig.suptitle(f"{caption} | ID={row.pair_id} | ảnh giả={row.fake_position}")
    plt.tight_layout()
    plt.show()

# Chọn ID nhỏ nhất trong nhóm lỗi, không chọn ảnh vì trông thuyết phục.
errors = va_fold0.loc[(p_32 >= 0.5) != y_val].copy()
errors['p_lr32'] = p_32[(p_32 >= 0.5) != y_val]
if errors.empty:
    print('Không có lỗi LR32 trên fold này; không tạo ví dụ thay thế.')
else:
    row = errors.sort_values('pair_id').iloc[0]
    show_case(row, f'LR32 p(ảnh 1 giả)={row.p_lr32:.3f}')

## Đọc bảng kết quả trước khi chọn bước tiếp theo

So sánh LR chỉ dùng dung lượng với quy tắc cố định: học hệ số từ train có giúp ích trên validation không? Sau đó so LR32 với LR31: bỏ dung lượng làm thay đổi bao nhiêu lỗi và Macro-F1? Sự khác biệt trên một fold là quan sát cần kiểm tra tiếp, chưa chứng minh cơ chế phát hiện ảnh giả. Một ảnh sai cũng không đủ để kết luận vì sao mô hình sai.

Notebook 02 chuyển sang học biểu diễn từ pixel. Nếu chỉ muốn hoàn thành baseline trên CPU, chuyển thẳng sang notebook 05 và giữ METHOD='lr'.